## Get illegal basements data

In [2]:
import os
import requests
import pandas as pd
from sodapy import Socrata
from dotenv import load_dotenv
load_dotenv()
os.environ['APP_TOKEN'][:8] + "..."

'Fikrpxj6...'

In [3]:
client = Socrata('data.cityofnewyork.us',
                 os.environ.get('APP_TOKEN'),
                 username=os.environ.get('USER_NAME'),
                 password=os.environ.get('PASS'),
                 timeout=100
                )

# First 12000 results, returned as JSON from API / converted to Python list of
# dictionaries by sodapy.
results = client.get("mr3a-22rc", limit=12000)
df = pd.DataFrame.from_records(results)
df.head(5)

,violationid,buildingid,registrationid,boroid,boro,housenumber,lowhousenumber,highhousenumber,streetname,streetcode,...,novid,novdescription,novissueddate,currentstatusid,currentstatus,currentstatusdate,certifieddate,apartment,newcertifybydate,newcorrectbydate
0,11760442,22593,0,1,MANHATTAN,8,8,8,HAMILTON TERRACE,22290,...,5740350,§ 27-2087 ADM CODE - DISCONTINUE USE OF ROOMS ...,2017-05-01T00:00:00.000,19,VIOLATION CLOSED,2024-03-26T00:00:00.000,NaN,NaN,NaN,NaN
1,13104700,258970,821908,3,BROOKLYN,862,862,862,EAST 34 STREET,36330,...,6505659,§ 27-2087 ADM CODE - DISCONTINUE USE OF ROOMS ...,2019-06-10T00:00:00.000,19,VIOLATION CLOSED,2024-11-08T00:00:00.000,NaN,NaN,NaN,NaN
2,14394274,529774,0,4,QUEENS,155-12,155-12,155-12,116 DRIVE,20900,...,7383980,§ 27-2087 ADM CODE - DISCONTINUE USE OF ROOMS ...,2021-06-21T00:00:00.000,19,VIOLATION CLOSED,2024-11-04T00:00:00.000,NaN,NaN,NaN,NaN
3,11788908,185356,811822,3,BROOKLYN,164,164,164,ATKINS AVENUE,13330,...,5756154,§ 27-2087 ADM CODE - DISCONTINUE USE OF ROOMS ...,2017-05-22T00:00:00.000,21,NOT COMPLIED WITH,2025-08-30T00:00:00.000,NaN,NaN,NaN,NaN
4,10777433,222651,826572,3,BROOKLYN,329,329,329,CLIFTON PLACE,27330,...,5157374,§ 27-2087 ADM CODE - DISCONTINUE USE OF ROOMS ...,2015-07-13T00:00:00.000,2,NOV SENT OUT,2015-07-13T00:00:00.000,NaN,NaN,NaN,NaN


In [ ]:
df.columns.tolist()

In [ ]:
df['class'].value_counts()

In [ ]:
df['novdescription'].unique()

In [ ]:
df['buildingid'].unique()

In [ ]:
print(df['block'].unique())
print(df['lot'].unique())

### Create BBL code

In [ ]:
df['block'] = df['block'].astype(str).str.zfill(5)
df['lot'] = df['lot'].astype(str).str.zfill(4)
df['BBL'] = df['boroid'].astype(str) + df['block'] + df['lot']
df

In [ ]:
df.to_csv('illegal_basement.csv', index=False)

In [ ]:
df['BBL'].unique()

### sort aftermath of Hurricane ida

In [ ]:
df['approveddate'] = pd.to_datetime(df['approveddate'])
df_approveddate_latest = (
    df
    .sort_values('approveddate')
    .groupby('BBL')
    .tail(1)
)
df_post_ida = df_approveddate_latest[
    df_approveddate_latest['approveddate'] >= '2021-09-01'
].copy()
df_post_ida

In [ ]:
df_post_ida.to_csv('illegal_basements_post_ida.csv', index=False)

### Classify “novdescription” using AI

In [183]:
import os
from IPython.display import display, Markdown
from tqdm.auto import tqdm # makes pretty progress bars
tqdm.pandas()              # makes pretty progress bars in pandas with progress_apply()

# this should be familiar!
from dotenv import load_dotenv
load_dotenv()
os.environ["OPENROUTER_API_KEY_LEDE"][:8] + "..."

'sk-or-v1...'

In [184]:
df_ai_classification = pd.read_csv('illegal_basements_post_ida.csv')
df_ai_classification = df_ai_classification.dropna(subset=['novdescription'])
text_column_name = 'novdescription'
df_ai_classification

,violationid,buildingid,registrationid,boroid,boro,housenumber,lowhousenumber,highhousenumber,streetname,streetcode,...,novdescription,novissueddate,currentstatusid,currentstatus,currentstatusdate,apartment,certifieddate,newcertifybydate,newcorrectbydate,BBL
0,14527976,70019,714507,2,BRONX,920,920,920,EAST 223 STREET,28320,...,§ 27-2087 ADM CODE - DISCONTINUE USE OF ROOMS ...,2021-09-07T00:00:00.000,21,NOT COMPLIED WITH,2022-01-21T00:00:00.000,CELLAR,NaN,NaN,NaN,2048580053
1,14528846,463180,0,4,QUEENS,54-37,54-37,54-37,69 LANE,14540,...,§ 27-2087 ADM CODE - DISCONTINUE USE OF ROOMS ...,2021-09-07T00:00:00.000,19,VIOLATION CLOSED,2022-02-18T00:00:00.000,NaN,NaN,NaN,NaN,4025030013
2,14537217,494525,0,4,QUEENS,24-03,24-03,24-03,87 STREET,17740,...,§ 27-2087 ADM CODE - DISCONTINUE USE OF ROOMS ...,2021-09-07T00:00:00.000,19,VIOLATION CLOSED,2021-09-07T00:00:00.000,NaN,NaN,NaN,NaN,4010990044
3,14537170,494527,0,4,QUEENS,24-05,24-05,24-05,87 STREET,17740,...,§ 27-2087 ADM CODE - DISCONTINUE USE OF ROOMS ...,2021-09-07T00:00:00.000,19,VIOLATION CLOSED,2021-09-07T00:00:00.000,NaN,NaN,NaN,NaN,4010990043
4,14537143,496476,0,4,QUEENS,24-25,24-25,24-25,88 STREET,17990,...,§ 27-2087 ADM CODE - DISCONTINUE USE OF ROOMS ...,NaN,19,VIOLATION CLOSED,2021-09-03T00:00:00.000,NaN,NaN,NaN,NaN,4011000063
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1650,19049414,593577,0,4,QUEENS,111-09,111-09,111-09,203 STREET,25990,...,§ 27-2087 ADM CODE - DISCONTINUE USE OF ROOMS ...,2026-07-06T00:00:00.000,2,NOV SENT OUT,2026-07-06T00:00:00.000,NaN,NaN,NaN,NaN,4109610546
1651,19055265,303705,0,3,BROOKLYN,1022,1022,1022,GREENE AVENUE,45930,...,§ 27-2087 ADM CODE - DISCONTINUE USE OF ROOMS ...,2026-07-06T00:00:00.000,2,NOV SENT OUT,2026-07-06T00:00:00.000,NaN,NaN,NaN,NaN,3016230018
1652,19017685,561592,922659,4,QUEENS,118-11,118-11,118-11,152 STREET,23380,...,§ 27-2087 ADM CODE - DISCONTINUE USE OF ROOMS ...,2026-07-06T00:00:00.000,2,NOV SENT OUT,2026-07-06T00:00:00.000,NaN,NaN,NaN,NaN,4122060014
1653,19047227,194689,0,3,BROOKLYN,1113,1113,1113,AVENUE T,14580,...,§ 27-2087 ADM CODE - DISCONTINUE USE OF ROOMS ...,2026-07-07T00:00:00.000,2,NOV SENT OUT,2026-07-07T00:00:00.000,NaN,NaN,NaN,NaN,3072900046


In [185]:
USE_OPENROUTER=True

from anthropic import Anthropic

if USE_OPENROUTER:
  from openrouter import OpenRouter
else:
  from openai import OpenAI
  from mistralai.client import Mistral

if USE_OPENROUTER:
  openrouter_client = OpenRouter(api_key=os.environ.get("OPENROUTER_API_KEY_LEDE"))
else:
  openai_client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
  mistral_client = Mistral(api_key=os.environ.get("MISTRAL_API_KEY"))
anthropic_client = Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"),)

In [186]:
## prompt
## start with the one YOU tried in the LLM web interface
## don't worry, we'll adjust it later.
## Jeremy's is at the very bottom, if you need to reference it, but please come
##   up with your own.


# Be sure to list the categories (defined below!) with the magic phrase {categories}
prompt_base = """

Please read this novdescription regarding the illegal basements in NYC.
You are expert and analyzing NYC housing violation descriptions.

IMPORTANT:

The phrase

"DISCONTINUE USE OF ROOMS FOR LIVING"

is standard enforcement language and
SHOULD NOT by itself increase risk_level
or conversion_level.

Focus instead on evidence such as:

- ILLEGAL CLASS A APARTMENT
- ILLEGAL BASEMENT APARTMENT
- KITCHEN CREATED
- BATHROOM CREATED
- REMOVE ALL WALLS
- CREATING ROOMS
- BEHIND WALLS
- UNDER FLOORS

- Use only information explicitly stated in the description.
- Do not speculate about the owner's intent.
- Do not infer motive, negligence, or malice.
- Do not assume facts not written in the text.
- Base classifications only on observable evidence in the violation text.

Classify the violation using the following framework.

For Risk Level:

low:
single plumbing fixture only

medium:
multiple residential fixtures suggesting habitation
(e.g. water closet, shower, wash basin and sink)

high:
illegal apartment explicitly referenced

critical:
illegal apartment plus evidence of conversion,
created kitchen/bathroom,
room creation,
or concealed work

For Conversion Evidence Level:

low:
plumbing fixtures only

medium:
multiple residential fixtures suggesting habitation,
but no explicit evidence of apartment creation

high:
illegal apartment explicitly referenced

critical:
explicit structural conversion,
created rooms,
created kitchen/bathroom,
walls removed,
concealed work behind walls or under floors

Evidence Flags:
- class_a_apartment
- cellar
- basement
- kitchen_created
- bathroom_created
- room_creation
- concealed_work

Definitions:
class_a_apartment:
true if the description explicitly contains:

"CLASS A APARTMENT"
"CLASS 'A' APARTMENT"
"CLASS ''A'' APARTMENT"

cellar:
true if the description explicitly mentions CELLAR.

basement:
true if the description explicitly mentions BASEMENT or BSMT.

kitchen_created:
true if the description explicitly states a kitchen was created or illegally created.

bathroom_created:
true if the description explicitly states a bathroom was created or illegally created.

room_creation:
true if the description mentions creating rooms, removing walls, or walls creating rooms.

concealed_work:
true if the description mentions work behind walls, under floors, underneath floors, concealed pipe connections, or similar concealed construction.

Return JSON only with the following exact keys:
{{
  "risk_level": "low|medium|high|critical",
  "conversion_level": "low|medium|high|critical",
  "class_a_apartment": true/false,
  "cellar": true/false,
  "basement": true/false,
  "kitchen_created": true/false,
  "bathroom_created": true/false,
  "room_creation": true/false,
  "concealed_work": true/false,
  "reason": "one short sentence citing exact wording from the description"
}}

Here is the novdescription:
{novdescription_text}

"""

from enum import Enum, IntEnum

# force the AI to ONLY guess fromn the options we ask you.
# you can change these names! (and perhaps should!)
# - the AI sees the _value_, not the key (it sees "broader_political_issues_except_us_election", not "other_politics")
# - the key is just for you -- a shorthand
# - some AIs have trouble with complicated names, so it's better to stick to lowercase, no special characters
#     (as of early June 2026 -- maybe this will be fixed)
class Illegal_basements_Options(str, Enum):
    low = 'low'
    medium = 'medium'
    high = 'high'
    critical = 'critical'
    
# for convenience, make a column in our dataframes with the prompt for each item
df_ai_classification_prompt_column = df_ai_classification.apply(lambda row: prompt_base.format(
    novdescription_text=row[text_column_name],
    categories=", ".join(['"{}"'.format(opt.value) for opt in Illegal_basements_Options])
), axis="columns")

In [187]:
df_ai_classification_prompt_column.iloc[500]

'\n\nPlease read this novdescription regarding the illegal basements in NYC.\nYou are expert and analyzing NYC housing violation descriptions.\n\nIMPORTANT:\n\nThe phrase\n\n"DISCONTINUE USE OF ROOMS FOR LIVING"\n\nis standard enforcement language and\nSHOULD NOT by itself increase risk_level\nor conversion_level.\n\nFocus instead on evidence such as:\n\n- ILLEGAL CLASS A APARTMENT\n- ILLEGAL BASEMENT APARTMENT\n- KITCHEN CREATED\n- BATHROOM CREATED\n- REMOVE ALL WALLS\n- CREATING ROOMS\n- BEHIND WALLS\n- UNDER FLOORS\n\n- Use only information explicitly stated in the description.\n- Do not speculate about the owner\'s intent.\n- Do not infer motive, negligence, or malice.\n- Do not assume facts not written in the text.\n- Base classifications only on observable evidence in the violation text.\n\nClassify the violation using the following framework.\n\nFor Risk Level:\n\nlow:\nsingle plumbing fixture only\n\nmedium:\nmultiple residential fixtures suggesting habitation\n(e.g. water clos

In [188]:
## cost estimation

from strip_tags import strip_tags
import tiktoken
def count_tokens(model, text):
  if "claude" in model and os.environ.get("ANTHROPIC_API_KEY_LEDE"):
    return anthropic_client.messages.count_tokens(model=model, messages=[{"content": text, "role": "user"}])
  elif "gpt" in model:
    encoding = tiktoken.encoding_for_model('gpt-4o') # most modern models use the same tokenizer
    tokens = encoding.encode(text)
    return len(tokens)
  else:
    return count_tokens("gpt-4o", text) # for estimation purposes, pretend mistral is chatgpt...

INPUT_TOKEN_COSTS = {
    "gpt-5-mini": 0.40 / 1_000_000,  # https://openai.com/api/pricing/
    "gpt-5.4-mini": 0.75 / 1_000_000,
    "gpt-5.4": 2.5 / 1_000_000,
    "gpt-5.5": 5 / 1_000_000,
    "gpt-5.4-nano": 0.2 / 1_000_000,
    "mistral-medium-latest": 1.5 / 1_000_000, # https://mistral.ai/pricing#api-pricing
    "mistral-large-latest":  0.5 / 1_000_000,  # No, I don't know why medium is more expensive than large.
    "claude-4.5-haiku": 1 / 1_000_000,
    "claude-4.6-sonnet": 3 / 1_000_000
}
# ignores outputs, since the model will output fairly little text

def estimate_cost(model, token_count):
    return token_count * INPUT_TOKEN_COSTS[model]

In [189]:
# IMPORTANT: let's pick a model to use
# it's got to be listed in INPUT_TOKEN_COSTS up above

DEFAULT_MODEL_TO_USE = 'gpt-5.4-nano'
display(Markdown("Pick a model to use: \n" + "\n - ".join(list(INPUT_TOKEN_COSTS.keys()))))
MODEL_TO_USE = None

while MODEL_TO_USE not in INPUT_TOKEN_COSTS.keys():
  MODEL_TO_USE = input(f"type a model name (press enter for default: {DEFAULT_MODEL_TO_USE}): ").strip()
  if MODEL_TO_USE == "":
    MODEL_TO_USE = DEFAULT_MODEL_TO_USE

Pick a model to use: 
gpt-5-mini
 - gpt-5.4-mini
 - gpt-5.4
 - gpt-5.5
 - gpt-5.4-nano
 - mistral-medium-latest
 - mistral-large-latest
 - claude-4.5-haiku
 - claude-4.6-sonnet

type a model name (press enter for default: gpt-5.4-nano):  gpt-5.4-mini


In [190]:
## cost estimation

count_tokens_for_our_model = lambda text: count_tokens(MODEL_TO_USE, text)

token_count_sample = count_tokens_for_our_model("SAMPLE RESPONSE SAMPLE RESPONSE".join(df_ai_classification_prompt_column))
"Sample would cost: ${:.2f}".format(estimate_cost(MODEL_TO_USE, token_count_sample))

'Sample would cost: $0.86'

In [199]:
## a lot of boilerplate for sending our prompt + tweet to the model and getting an answer
import re
from pydantic import BaseModel

class Illegal_basements_ValidOptions(BaseModel):
    risk_level: str
    conversion_level: str
    class_a_apartment: bool
    cellar: bool
    basement: bool
    kitchen_created: bool
    bathroom_created: bool
    room_creation: bool
    concealed_work: bool
    reason: str
  
SYSTEM_PROMPT = "You are a specialist of illegal basemets in NYC"

def anthropic_classify(prompt_including_comment):
    # put our prompt into the blob that OpenAI expects
    message = anthropic_client.messages.parse(
        max_tokens=1024,
        # system=SYSTEM_PROMPT, # causes trouble with structured outputs, oddly
        messages = [
            {
                "role": "user",
                "content": prompt_including_comment,
            }
        ],
        model=MODEL_TO_USE,
        output_format=Illegal_basements_ValidOptions
    )
    return message.content[0].parsed_output.classification.value if message.content[0].parsed_output else None

def mistral_classify(prompt_including_comment):
    # put our prompt into the blob that OpenAI expects
    chat_response = mistral_client.chat.parse(
      model=MODEL_TO_USE,
      messages=[
          {
              "role": "system",
              "content": SYSTEM_PROMPT
          },
          {
              "role": "user",
              "content": prompt_including_comment
          },
      ],
      response_format=Illegal_basements_ValidOptions,
      max_tokens=256,
      temperature=0
    )
    return chat_response.choices[0].message.parsed.classification.value if chat_response.choices[0].message.parsed else None

def chatgpt_classify(prompt_including_comment):
    # put our prompt into the blob that OpenAI expects
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": prompt_including_comment,
        }
    ]
    chat_completion = openai_client.responses.parse(
        input=messages,
        model=MODEL_TO_USE,
        text_format=Illegal_basements_ValidOptions,
    )

    # get the answer out of the blob that OpenAI returns.
    resp = chat_completion.output_parsed.classification.value if chat_completion.output_parsed else None
    return resp

openrouter_model_names = {
    "gpt-5-mini": "openai/gpt-5-mini",
    "gpt-5.4-mini": "openai/gpt-5.4-mini",
    "gpt-5.4": "openai/gpt-5.4",
    "gpt-5.5": "openai/gpt-5.5",
    "gpt-5.4-nano": "openai/gpt-5.4-nano",
    "mistral-medium-latest": "mistralai/mistral-medium-3-5",
    "mistral-large-latest": "mistralai/mistral-large-2512",
    "claude-4.5-haiku": "anthropic/claude-4.5-haiku",
    "claude-4.6-sonnet": "anthropic/claude-4.6-sonnet"
}
def openrouter_classify(prompt_including_comment):
    response = openrouter_client.chat.send(
        model=openrouter_model_names[MODEL_TO_USE],
        messages=[
            {"role": "user", "content": prompt_including_comment}
        ],
        temperature=0
    )

    raw_json = response.choices[0].message.content

    # 末尾カンマ対策
    raw_json = re.sub(
        r",\s*([}\]])",
        r"\1",
        raw_json
    )

    result = Illegal_basements_ValidOptions.model_validate_json(
        raw_json
    )

    return result.model_dump()
    
def classify(prompt_including_comment):
    "A wrapper for the provider-specific classification functions"

    try:

        if USE_OPENROUTER:
            return openrouter_classify(prompt_including_comment)

        else:
            if "mistral" in MODEL_TO_USE:
                return mistral_classify(prompt_including_comment)

            elif "claude" in MODEL_TO_USE:
                return anthropic_classify(prompt_including_comment)

            elif "gpt" in MODEL_TO_USE:
                return chatgpt_classify(prompt_including_comment)

            else:
                raise ValueError("Unknown model")

    except Exception as e:

        return {
            "risk_level": None,
            "conversion_level": None,
            "class_a_apartment": None,
            "cellar": None,
            "basement": None,
            "kitchen_created": None,
            "bathroom_created": None,
            "room_creation": None,
            "concealed_work": None,
            "reason": f"ERROR: {e}"
        }

In [139]:
test = classify(
    df_ai_classification_prompt_column.iloc[0]
)

print(test)
print(type(test))

{'risk_level': 'low', 'conversion_level': 'low', 'class_a_apartment': False, 'cellar': True, 'basement': False, 'kitchen_created': False, 'bathroom_created': False, 'room_creation': False, 'concealed_work': False, 'reason': 'The description only says "DISCONTINUE USE OF ROOMS FOR LIVING" and "DISCONNECT PLUMBING FIXTURES" in the "CELLAR APT CELLAR," with no explicit illegal apartment or conversion language.'}
<class 'dict'>


In [200]:
results = df_ai_classification_prompt_column.progress_apply(classify)

results_df = pd.DataFrame(results.tolist())

df_ai_classification = pd.concat(
    [df_ai_classification, results_df],
    axis=1
)

100%|███████████████████████████████████████| 1655/1655 [58:12<00:00,  2.11s/it]


In [205]:
df_ai_classification.head()
df_ai_classification.to_csv('illegal_basements_ai_classification.csv', index=False)

In [203]:
print(df_ai_classification['risk_level'].value_counts())
print(df_ai_classification['conversion_level'].value_counts())
print(df_ai_classification['class_a_apartment'].value_counts())

risk_level
medium      1110
critical     246
high         200
low           99
Name: count, dtype: int64
conversion_level
medium      715
low         487
critical    258
high        195
Name: count, dtype: int64
class_a_apartment
False    1391
True      264
Name: count, dtype: int64


In [6]:
df_ai_classification.columns.tolist()

NameError: name 'df_ai_classification' is not defined

In [7]:
basements_flood_post_ida = pd.read_csv('basements_flood_post_ida.csv')
basements_flood_post_ida

,Borough,Block,Lot,CD,BCT2020,BCTCB2020,CT2010,CB2010,SchoolDist,Council,...,streetname,zip,illegal_count,year,min_elev,mean_elve,elev_class,flood_current,flood_2050,flood_2080
0,MN,375,41,103,1002601,10026011000,26.01,1000,1,2,...,AVENUE D,10009.0,1.0,2026.0,7.0,7.0,5-10 ft,False,False,False
1,MN,1142,43,107,1015700,10157008000,157.00,8000,3,6,...,WEST 71 STREET,10023.0,1.0,2025.0,79.0,79.0,20 ft+,False,False,False
2,BX,2278,25,201,2001902,20019021033,19.00,1010,7,8,...,EAST 134 STREET,10454.0,1.0,2023.0,38.0,38.0,20 ft+,False,False,False
3,BX,2288,73,201,2004100,20041002000,41.00,2000,7,8,...,EAST 143 STREET,10454.0,1.0,2024.0,13.0,13.0,10-20 ft,False,False,False
4,BX,2433,121,204,2017500,20175002000,175.00,2000,9,16,...,EAST 166 STREET,10456.0,1.0,2025.0,31.0,31.0,20 ft+,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1648,SI,4278,46,503,5013201,50132011009,132.01,1010,31,50,...,TYSENS LANE,10306.0,1.0,2022.0,17.0,17.0,10-20 ft,True,True,True
1649,SI,4310,128,503,5013800,50138005001,138.00,5001,31,50,...,DOROTHEA PLACE,10306.0,3.0,2022.0,80.0,80.0,20 ft+,False,False,False
1650,SI,4478,11,503,5014606,50146063001,146.06,3001,31,50,...,CORONA AVENUE,10306.0,1.0,2023.0,67.0,67.0,20 ft+,True,True,True
1651,SI,6334,32,503,5017009,50170092012,170.09,2020,31,51,...,IONIA AVENUE,10312.0,1.0,2023.0,118.0,118.0,20 ft+,False,False,False


In [8]:
df_ai_classification = pd.read_csv('illegal_basements_ai_classification.csv')
basements_total_post_ida = pd.merge(basements_flood_post_ida, df_ai_classification,
                                    on='BBL',
                                    how='left',
                                    indicator=True
                                   )
basements_total_post_ida["_merge"].value_counts()

/var/folders/jp/l9xrflms6x1g316qkkbvxp9w0000gq/T/ipykernel_3008/1878647558.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  basements_total_post_ida = pd.merge(basements_flood_post_ida, df_ai_classification,


_merge
both          1653
left_only        0
right_only       0
Name: count, dtype: int64

In [12]:
basements_total_post_ida.columns.tolist()

['Borough',
 'Block',
 'Lot',
 'CD',
 'BCT2020',
 'BCTCB2020',
 'CT2010',
 'CB2010',
 'SchoolDist',
 'Council',
 'ZipCode',
 'FireComp',
 'PolicePrct',
 'HealthCent',
 'HealthArea',
 'Sanitboro',
 'SanitDistr',
 'SanitSub',
 'Address',
 'ZoneDist1',
 'ZoneDist2',
 'ZoneDist3',
 'ZoneDist4',
 'Overlay1',
 'Overlay2',
 'SPDist1',
 'SPDist2',
 'SPDist3',
 'LtdHeight',
 'SplitZone',
 'BldgClass',
 'LandUse',
 'Easements',
 'OwnerType',
 'OwnerName',
 'LotArea',
 'BldgArea',
 'ComArea',
 'ResArea',
 'OfficeArea',
 'RetailArea',
 'GarageArea',
 'StrgeArea',
 'FactryArea',
 'OtherArea',
 'AreaSource',
 'NumBldgs',
 'NumFloors',
 'UnitsRes',
 'UnitsTotal',
 'LotFront',
 'LotDepth',
 'BldgFront',
 'BldgDepth',
 'Ext',
 'ProxCode',
 'IrrLotCode',
 'LotType',
 'BsmtCode',
 'AssessLand',
 'AssessTot',
 'ExemptTot',
 'YearBuilt',
 'YearAlter1',
 'YearAlter2',
 'HistDist',
 'Landmark',
 'BuiltFAR',
 'ResidFAR',
 'CommFAR',
 'FacilFAR',
 'BoroCode',
 'BBL',
 'CondoNo',
 'Tract2010',
 'XCoord',
 'YCoo

In [18]:
basements_total_post_ida['boro_x'].value_counts()

boro_x
QUEENS           705
BROOKLYN         467
BRONX            393
STATEN ISLAND     84
MANHATTAN          4
Name: count, dtype: int64

In [14]:
basements_total_post_ida['boro_x'].value_counts(normalize=True)

boro_x
QUEENS           0.426497
BROOKLYN         0.282517
BRONX            0.237750
STATEN ISLAND    0.050817
MANHATTAN        0.002420
Name: proportion, dtype: float64

In [16]:
print(basements_total_post_ida['conversion_level'].value_counts())
print(basements_total_post_ida['risk_level'].value_counts())

conversion_level
medium      715
low         487
critical    257
high        194
Name: count, dtype: int64
risk_level
medium      1110
critical     245
high         199
low           99
Name: count, dtype: int64


In [17]:
print(basements_total_post_ida['conversion_level'].value_counts(normalize=True))
print(basements_total_post_ida['risk_level'].value_counts(normalize=True))

conversion_level
medium      0.432547
low         0.294616
critical    0.155475
high        0.117362
Name: proportion, dtype: float64
risk_level
medium      0.671506
critical    0.148215
high        0.120387
low         0.059891
Name: proportion, dtype: float64
